# Getting Started with LZGraphs

This notebook covers the fundamentals:
- Building a graph from sequences
- Computing LZPGEN (generation probability)
- Simulating new sequences
- Basic graph properties and diagnostics

In [ ]:
from LZGraphs import LZGraph, lz76_decompose
import numpy as np

## 1. LZ76 Decomposition

LZGraphs is built on the **Lempel-Ziv 76** compression algorithm. Each CDR3 sequence is decomposed into subpatterns where each new token extends a previously seen one by one character.

In [ ]:
tokens = lz76_decompose('CASSLEPSGGTDTQYF')
print(f'Tokens: {tokens}')
print(f'Count:  {len(tokens)}')

## 2. Building a Graph

The `LZGraph` class builds a directed acyclic graph from a list of sequences. Each node is an LZ76 subpattern (with positional encoding for AAP variant), and edges represent consecutive tokens in the decomposition.

In [ ]:
# Load example data (5000 CDR3 amino acid sequences with V/J genes)
import csv

sequences, v_genes, j_genes = [], [], []
with open('ExampleData3.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sequences.append(row['cdr3_amino_acid'])
        v_genes.append(row['V'])
        j_genes.append(row['J'])

print(f'{len(sequences)} sequences loaded')
print(f'First 3: {sequences[:3]}')

In [ ]:
# Build the graph with gene annotations
graph = LZGraph(
    sequences,
    variant='aap',
    v_genes=v_genes,
    j_genes=j_genes,
)
print(graph)

In [ ]:
# Basic properties
print(f'Nodes:         {graph.n_nodes}')
print(f'Edges:         {graph.n_edges}')
print(f'Variant:       {graph.variant}')
print(f'Has gene data: {graph.has_gene_data}')
print(f'Is DAG:        {graph.is_dag}')
print(f'Path count:    {graph.path_count}')

In [ ]:
# Full structural summary
graph.summary()

## 3. LZPGEN — Sequence Probability

The core computation: given a sequence, what is its probability under the graph model?

`lzpgen()` re-encodes the sequence via LZ76, traces the canonical walk through the graph, and returns the product of renormalized edge weights at each step — fully respecting LZ76 dictionary constraints.

In [ ]:
# Single sequence (log-probability by default)
lp = graph.lzpgen('CASSLEPSGGTDTQYF')
print(f'log P(gen) = {lp:.4f}')

# Probability (not log)
p = graph.lzpgen('CASSLEPSGGTDTQYF', log=False)
print(f'P(gen)     = {p:.2e}')

In [ ]:
# Batch scoring — pass a list, get a numpy array
test_seqs = sequences[:10]
log_probs = graph.lzpgen(test_seqs)
for seq, lp in zip(test_seqs, log_probs):
    print(f'{seq:25s}  log P = {lp:.4f}')

In [ ]:
# Check membership (True if sequence has positive probability)
print(f'"CASSLEPSGGTDTQYF" in graph: {"CASSLEPSGGTDTQYF" in graph}')
print(f'"XXXXXXXXXX" in graph:       {"XXXXXXXXXX" in graph}')

## 4. Simulation

Generate new sequences by random walks through the graph. Every walk respects LZ76 dictionary constraints and is guaranteed to terminate at a valid terminal state.

In [ ]:
# Simulate 10 sequences
result = graph.simulate(10, seed=42)

# SimulationResult is iterable
for seq in result:
    print(seq)

In [ ]:
# Access metadata
print(f'Log-probabilities: {result.log_probs}')
print(f'Token counts:      {result.n_tokens}')

In [ ]:
# Gene-constrained simulation
result_genes = graph.simulate(5, seed=42, v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01')
for seq, v, j in zip(result_genes.sequences, result_genes.v_genes, result_genes.j_genes):
    print(f'{seq:25s}  V={v}  J={j}')

## 5. Model Diagnostics

Verify that the graph defines a proper probability distribution (total mass = 1.0).

In [ ]:
diag = graph.pgen_diagnostics()
print(f'Total absorbed mass: {diag["total_absorbed"]:.10f}')
print(f'Total leaked:        {diag["total_leaked"]:.2e}')
print(f'Is proper:           {diag["is_proper"]}')

## 6. Save & Load

Graphs are saved in the LZG binary format (`.lzg`) with CRC-32C integrity checks.

In [ ]:
graph.save('my_graph.lzg')

loaded = LZGraph.load('my_graph.lzg')
print(f'Loaded: {loaded}')
print(f'Gene data preserved: {loaded.has_gene_data}')

import os
os.remove('my_graph.lzg')